### 1. Settings

Paths and settings for the tests. Set `QUICK_TEST = False` to run the full evaluation.

In [ ]:
from pathlib import Path
from copy import deepcopy
from functools import lru_cache
from itertools import product
from contextlib import contextmanager
import io
import json
import os
import signal
import sys
import time
import urllib.request
import zipfile
import numpy as np

PROJECT = Path("/var/tmp/schmidtn-diffusion/Study-Project")
DATA = PROJECT / "own_code/demos"
MODELS = PROJECT / "own_code/models"
QUICK_TEST = True
TEST_ACTIONS = 3
TEST_SECONDS = 30
MAX_ACTIONS = 400
MAX_SECONDS = 600
OUTPUT = PROJECT / "own_code" / (
    f"results_smoke_{TEST_ACTIONS}a_{TEST_SECONDS}s" if QUICK_TEST
    else f"results_notebook_{MAX_ACTIONS}a_{MAX_SECONDS}s")

CASE_SEED = 29092026
SOURCE = "https://raw.githubusercontent.com/Nils19024/Study-Project/6e6473895af1ebb0798bfc021c348160bfc423d9/own_code/results"
RELEASE = "https://github.com/Nils19024/Study-Project/releases/download/evaluation-models"
TASKS = {
    "push_button": ("BimanualDualPushButtons", "bimanual_dual_push_buttons", ["push_buttons_target0", "push_buttons_target1"]),
    "hand_over": ("BimanualHandoverItem", "bimanual_handover_item", ["item0"]),
    "put_bottle_in_fridge": ("BimanualPutBottleInFridge", "bimanual_put_bottle_in_fridge", ["bottle"]),
    "take_tray_out_of_oven": ("BimanualTakeTrayOutOfOven", "bimanual_take_tray_out_of_oven", ["oven_boundary_root"]),
}
MODEL_NAMES = ["joint", "left_leader", "right_leader", "tapas"]
timings = []
diagnostics = []


### 2. Imports

Imports for RLBench, TAPAS and the model code.

In [ ]:
sys.path.insert(0, str(PROJECT / "TAPAS"))
os.environ.setdefault("OMP_NUM_THREADS", "1")
import torch
from loguru import logger
from pyrep.objects.object import Object
from pyrep.objects.shape import Shape
from pyrep.const import ObjectType
from pyrep.errors import IKError, ConfigurationPathError
from rlbench.backend.exceptions import InvalidActionError
from rlbench.action_modes.arm_action_modes import BimanualEndEffectorPoseViaIK, BimanualEndEffectorPoseViaPlanning
from tapas_gmm.env.rlbench import RLBenchEnvironment, RLBenchEnvironmentConfig
from tapas_gmm.policy.diffusion import DiffusionPolicy
from tapas_gmm.policy.gmm import GMMPolicy, GMMPolicyConfig
from tapas_gmm.policy.models.tpgmm import AutoTPGMMConfig
from tapas_gmm.encoder.encoder import ObservationEncoderConfig
from tapas_gmm.utils.select_gpu import device
from conf.behavior_cloning.diffusion.five_demos.default import config

torch.set_num_threads(1)
logger.remove()
assert PROJECT.is_dir()
assert OUTPUT.resolve() != (PROJECT / "own_code/results").resolve()


### 3. Files and checkpoints

Functions for loading checkpoints and saving the results as JSON. Missing models are downloaded from the GitHub release.

In [ ]:
@lru_cache(None)
def reference(folder, task, model):
    url = f"{SOURCE}/{folder}/{task}/{model}.json"
    with urllib.request.urlopen(url, timeout=60) as response:
        return json.load(response)

def save(folder, task, model, parameters, records):
    path = OUTPUT / folder / task / f"{model}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    data = {"parameters": parameters, "records": records}
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(data, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)
    return path

def checkpoints(task, model, demos, seed, epoch):
    variant = {"left_leader": "lf_left", "right_leader": "lf_right",
               "tapas": "tapas_gripper", "tapas_velocity": "tapas_velocity"}.get(model, model)
    directory = MODELS / f"{task}_{variant}_{demos}demos"
    if model.startswith("tapas"):
        names = [f"seed{seed}/checkpoint.pkl"]
    elif model == "joint":
        names = [f"epoch{epoch}/seed{seed}/joint_ema.pt"]
    else:
        names = [f"epoch{epoch}/seed{seed}/{arm}_ema.pt" for arm in ["leader", "follower"]]
    paths = [directory / name for name in names]
    if not all(path.exists() for path in paths):
        with urllib.request.urlopen(f"{RELEASE}/{directory.name}.zip", timeout=120) as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
        for name in archive.namelist():
            target = directory / name
            assert target.resolve().is_relative_to(directory.resolve())
            if not name.endswith("/") and not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                partial = target.with_suffix(target.suffix + ".part")
                partial.write_bytes(archive.read(name))
                partial.replace(target)
    assert all(path.is_file() for path in paths), paths
    return paths


### 4. Load the models

Model loading and action prediction for Joint, leader-follower and TAPAS.

In [ ]:
@lru_cache(1)
def load_policies(task, model, demos, seed, epoch, k):
    paths = checkpoints(task, model, demos, seed, epoch)
    if model.startswith("tapas"):
        policy = GMMPolicy(GMMPolicyConfig(
            suffix=None, model=AutoTPGMMConfig(), batch_predict_in_t_models=False,
            topp_in_t_models=False, dbg_prediction=False, binary_gripper_action=True,
            force_overwrite_checkpoint_config=True, pos_lag_thresh=0.03, quat_change_thresh=0.1))
        policy.from_disk(str(paths[0]))
        cfg = policy.model.config
        assert policy.model._demos.is_bimanual
        assert cfg.tpgmm.n_components == 10 and cfg.tpgmm.model_type.name == "HMM"
        assert cfg.tpgmm.add_time_component and cfg.tpgmm.add_gripper_action
        assert not cfg.tpgmm.position_only
        assert cfg.demos_segmentation.components_prop_to_len
        assert cfg.demos_segmentation.min_n_components == 3
        assert cfg.demos_segmentation.velocity_based == (model == "tapas_velocity")
        assert cfg.demos_segmentation.gripper_based == (model == "tapas")
        assert cfg.frame_selection.rel_score_threshold == 0.75
        assert cfg.frame_selection.drop_redundant_frames
        assert all(segment.fitting_stage.name == "EM_HMM" for segment in policy.model.segment_gmms)
        return [policy.eval()]
    policies = []
    leader = "right" if model == "right_leader" else "left"
    for index, path in enumerate(paths):
        state = torch.load(path, map_location="cpu")
        cfg = deepcopy(config.policy)
        cfg.training = None
        cfg.obs_dim = state["normalizer.params_dict.obs.offset"].numel()
        cfg.action_dim = 16 if model == "joint" else 8
        cfg.arm = None if model == "joint" else (leader if index == 0 else {"left": "right", "right": "left"}[leader])
        cfg.condition_on_arm = leader if index == 1 else None
        cfg.horizon, cfg.n_obs_steps, cfg.n_action_steps = 16, 2, k
        cfg.num_inference_steps = 100
        cfg.obs_encoder = ObservationEncoderConfig(ee_pose=True, object_poses=True)
        cfg.unet.input_dim, cfg.unet.down_dims = cfg.action_dim, (64, 128, 256)
        cfg.unet.global_cond_dim = cfg.obs_dim * 2 + (128 if index == 1 else 0)
        assert cfg.scheduler.num_train_timesteps == 100
        assert cfg.scheduler.beta_schedule == "squaredcos_cap_v2"
        policy = DiffusionPolicy(cfg).to(device)
        policy.load_state_dict(state)
        policy.to(device)
        policy.observation_padding = "zero" if task == "push_button" else "repeat_first"
        policies.append(policy.eval())
    return policies

def predict(policies, observation, model):
    first, info = policies[0].predict(observation)
    if model.startswith("tapas"):
        return np.array([first]), info.get("done", False)
    if model == "joint":
        return np.column_stack((first.ee, first.gripper)), False
    second, _ = policies[1].predict(observation, condition=info["action_pred"])
    left, right = (second, first) if model == "right_leader" else (first, second)
    return np.column_stack((left.ee, right.ee, left.gripper, right.gripper)), False


### 5. Simulation setup

RLBench setup, time limits and action execution.

In [ ]:
class InvalidStart(Exception):
    pass

class Deadline(BaseException):
    pass

@contextmanager
def deadline(seconds):
    def expired(signum, frame):
        raise Deadline()
    previous = signal.signal(signal.SIGALRM, expired)
    signal.setitimer(signal.ITIMER_REAL, seconds)
    try:
        yield
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, previous)

def environment(task, model):
    tapas = model.startswith("tapas")
    planning = tapas or task == "hand_over"
    return RLBenchEnvironment(RLBenchEnvironmentConfig(
        task=TASKS[task][0], robot_setup="dual_panda",
        action_mode=BimanualEndEffectorPoseViaPlanning if planning else BimanualEndEffectorPoseViaIK,
        cameras=(), camera_pose={}, image_size=(128, 128), static=False, headless=True,
        scale_action=False, delay_gripper=False, gripper_plot=False,
        absolute_action_mode=not tapas, action_frame="end effector" if tapas else "world"))

def execute(env, action):
    native = []
    for side in (1, 0):
        command = np.r_[action[7 * side:7 * side + 7], action[14 + side]]
        command = env.postprocess_action(command, scale_action=False, delay_gripper=False, prediction_is_quat=True)
        native.extend([*command[:7], float(command[7] > 0), 0.0])
    try:
        raw, reward, done = env.task_env.step(np.array(native))
    except (IKError, InvalidActionError, ConfigurationPathError):
        raw = env.task_env.get_observation()
        reward, done = env.task_env._task.success()
    return env.process_observation(raw), bool(reward), bool(done)



### 6. Start positions and impulses

Functions to shift the start positions and apply an impulse when a hand is close to the object.

In [ ]:
def shift_start(env, task, episode, distance):
    names = TASKS[task][2]
    if task == "push_button":
        path = checkpoints(task, "joint", 100, 1, 2000)[0]
        state = torch.load(path, map_location="cpu")
        bounds = [state[f"normalizer.params_dict.obs.input_stats.{key}"][14:].reshape(3, 7)[:2, :2].numpy()
                  for key in ("min", "max")]
    else:
        files = sorted((DATA / TASKS[task][1]).glob("trajectories/*/object_poses/0.dat"))
        assert len(files) == 100
        objects = [obj.get_name() for obj, _ in env.task_env._task._initial_objs_in_scene]
        keys = [f"obj{objects.index(name):03d}" for name in names]
        starts = np.array([[torch.load(p, map_location="cpu")[key][:2].numpy() for key in keys] for p in files])
        bounds = [starts.min(axis=0), starts.max(axis=0)]
    axis, sign = [(0, 1), (0, -1), (1, 1), (1, -1)][episode % 4]
    placed = []
    for index, name in enumerate(names):
        obj = Object.get_object(name)
        position = obj.get_position()
        position[axis] = bounds[int(sign > 0)][index, axis] + sign * distance / 100
        obj.set_position(position)
        placed.append((obj, position.copy()))
    def bounds(obj):
        box, matrix = obj.get_bounding_box(), obj.get_matrix()
        corners = np.array(list(product(*[box[i:i + 2] for i in (0, 2, 4)])))
        points = corners @ matrix[:3, :3].T + matrix[:3, 3]
        return points.min(axis=0)[:2], points.max(axis=0)[:2]
    low, high = bounds(Object.get_object("diningTable"))
    for obj, _ in placed:
        bottom, top = bounds(obj)
        if np.any(bottom < low) or np.any(top > high):
            raise InvalidStart("Object outside the tabletop")
    robot = env.env._robot
    shapes = env.task_env._pyrep.get_objects_in_tree(object_type=ObjectType.SHAPE)
    for arm in [robot.left_arm, robot.right_arm]:
        own = {shape.get_handle() for shape in arm.get_objects_in_tree(object_type=ObjectType.SHAPE, exclude_base=False)}
        if any(shape.is_collidable() and shape.get_handle() not in own and arm.check_arm_collision(shape) for shape in shapes):
            raise InvalidStart("Object intersects the robot")
    for _ in range(10):
        env.task_env._pyrep.step()
    if any(np.linalg.norm(obj.get_position() - position) > 0.01 for obj, position in placed):
        raise InvalidStart("Object moved while settling")
    if env.task_env._task.success()[0]:
        raise InvalidStart("Task already successful at the start")
    return env.process_observation(env.task_env.get_observation())

def install_impulse(env, task, episode, magnitude, result):
    target = Shape({"hand_over": "item0", "put_bottle_in_fridge": "bottle", "take_tray_out_of_oven": "tray"}[task])
    assert target.is_dynamic()
    simulation, robot = env.task_env._pyrep, env.env._robot
    original_step = simulation.step
    direction = np.array([(1, 0, 0), (-1, 0, 0), (0, 1, 0), (0, -1, 0)][episode % 4])
    result.update(impulse_applied=False, force_n=None, impulse_duration_seconds=None)
    def step(*args, **kwargs):
        grippers = [robot.left_gripper, robot.right_gripper]
        tips = [robot.left_arm.get_tip(), robot.right_arm.get_tip()]
        box = np.array(target.get_bounding_box()).reshape(3, 2)
        points = [tip.get_position(relative_to=target) for tip in tips]
        near = min(np.linalg.norm(point - np.clip(point, box[:, 0], box[:, 1])) for point in points) < 0.10
        applied_now = not result["impulse_applied"] and near and all(target not in gripper.get_grasped_objects() for gripper in grippers)
        if applied_now:
            dt = simulation.get_simulation_timestep()
            force = (direction * magnitude / dt).tolist()
            target.add_force_and_torque(force=force, torque=None)
            result.update(impulse_applied=True, force_n=force, impulse_duration_seconds=dt)
        value = original_step(*args, **kwargs)
        if applied_now and env.task_env._task.success()[0]:
            result["invalid_intervention"] = True
        return value
    simulation.step = step
    return original_step


### 7. One episode

Run one episode and collect its result, number of actions and runtime.

In [ ]:
def episode(task, model, parameters, folder):
    demos, seed, epoch, k = [parameters[key] for key in ["demos", "seed", "epoch", "k"]]
    number = parameters["episode"]
    checkpoints(task, model, demos, seed, epoch)
    load_policies.cache_clear()
    started = time.monotonic()
    policies = load_policies(task, model, demos, seed, epoch, k)
    np.random.seed(CASE_SEED + number)
    torch.manual_seed(CASE_SEED + number + 1000 * seed)
    rng = np.random.default_rng(CASE_SEED + number)
    bias = rng.normal(size=(2, 3))
    bias /= np.linalg.norm(bias, axis=1, keepdims=True)
    result = {"status": "error", "success": False, "actions": 0}
    env, original_step = None, None
    if folder == "impulse":
        result.update(impulse_applied=False, force_n=None, impulse_duration_seconds=None)
    try:
        with deadline(TEST_SECONDS if QUICK_TEST else MAX_SECONDS):
            env = environment(task, model)
            observation = env.reset()
            if env.task_env._task.success()[0]:
                raise InvalidStart("Task already successful at reset")
            if folder == "position":
                observation = shift_start(env, task, number, parameters["position_shift_cm"])
            if folder == "impulse":
                original_step = install_impulse(env, task, number, parameters["impulse_ns"], result)
            for policy in policies:
                policy.reset_episode(env)
                if not model.startswith("tapas") and task != "push_button":
                    policy._obs_que.append(observation)
            limit = TEST_ACTIONS if QUICK_TEST else MAX_ACTIONS
            finished = False
            with torch.inference_mode():
                while result["actions"] < limit and not finished:
                    actions, policy_done = predict(policies, observation, model)
                    for index, action in enumerate(actions):
                        action = np.array(action, copy=True)
                        if folder.startswith("offset"):
                            if folder == "offset_random":
                                bias = rng.normal(size=(2, 3))
                                bias /= np.linalg.norm(bias, axis=1, keepdims=True)
                            for side, arm in enumerate([env.env._robot.left_arm, env.env._robot.right_arm]):
                                delta = bias[side] * parameters["offset_cm"] / 100
                                if model.startswith("tapas"):
                                    delta = arm.get_tip().get_matrix()[:3, :3].T @ delta
                                action[7 * side:7 * side + 3] += delta
                        observation, success, done = execute(env, action)
                        result["actions"] += 1
                        result["success"] |= success
                        finished = success or done or policy_done
                        if finished or result["actions"] == limit:
                            break
                        if index < len(actions) - 1 and not model.startswith("tapas"):
                            for policy in policies:
                                policy._obs_que.append(observation)
            result["status"] = "completed" if finished or not QUICK_TEST else "smoke_limit"
            if folder == "impulse" and not result["impulse_applied"]:
                result["status"] = "not_triggered"
            if result.get("invalid_intervention"):
                result["status"] = "invalid_intervention"
    except Deadline:
        result["status"] = "timeout"
    except InvalidStart as error:
        result.update(status="invalid_start", error=str(error))
    except Exception as error:
        result.update(status="error", error=f"{type(error).__name__}: {error}")
    finally:
        if env is not None:
            if original_step is not None:
                env.task_env._pyrep.step = original_step
            env.close()
    result["wall_seconds"] = time.monotonic() - started
    timings.append(dict(task=task, model=model, folder=folder, **parameters, **result))
    if result["status"] != "completed":
        diagnostics.append(timings[-1])
    return result


### 8. Run an evaluation block

Run the test configurations and save the results after each episode. Already saved episodes are skipped.

In [ ]:
def run(folder):
    summary = []
    tasks = list(TASKS)
    models = ["tapas_velocity"] if folder == "segmentation_comparison" else MODEL_NAMES
    if folder == "impulse":
        tasks.remove("push_button")
    for task, model in product(tasks, models):
        template = reference(folder, task, model)
        rows = template["records"]
        if QUICK_TEST:
            if task != "hand_over" or model not in ["joint", "tapas", "tapas_velocity"]:
                continue
            rows = rows[:1]
        path = OUTPUT / folder / task / f"{model}.json"
        records = []
        if path.exists():
            previous = json.loads(path.read_text())
            assert previous["parameters"] == template["parameters"]
            records = previous["records"]
            assert len(records) <= len(rows)
        for index, row in enumerate(rows):
            parameters = {**template["parameters"], **{key: value for key, value in row.items()
                if key in ["demos", "epoch", "k", "seed", "episode", "offset_cm", "position_shift_cm", "impulse_ns"]}}
            if index < len(records):
                stored = {**template["parameters"], **records[index]}
                assert all(stored[key] == value for key, value in parameters.items())
                expected = set(row) | ({"actions", "wall_seconds"} if folder == "baseline" else set())
                assert set(records[index]) == expected
                continue
            result = episode(task, model, parameters, folder)
            values = {**parameters, **result}
            if "training_minutes" in row:
                values["training_minutes"] = row["training_minutes"]
            record = {key: values[key] for key in row}
            if folder == "baseline":
                record.update(actions=result["actions"], wall_seconds=result["wall_seconds"])
            records.append(record)
            save(folder, task, model, template["parameters"], records)
        summary.append({"file": str(path.relative_to(OUTPUT)), "records": len(records),
                        "statuses": {status: sum(row["status"] == status for row in records)
                                     for status in sorted({row["status"] for row in records})}})
    return summary


### 9. Replanning functions

Functions for the replanning test and plot. The TAPAS prediction is called on the first simulation step after the impulse.

In [ ]:
def simulation_counter(env):
    simulation = env.task_env._pyrep
    original = simulation.step
    counter = {"steps": 0}
    def step(*args, **kwargs):
        value = original(*args, **kwargs)
        counter["steps"] += 1
        return value
    simulation.step = step
    return counter, original

def initialize(env, policies):
    np.random.seed(CASE_SEED)
    torch.manual_seed(CASE_SEED + 1000)
    observation = env.reset()
    for policy in policies:
        policy.reset_episode(env)
        if hasattr(policy, "_obs_que"):
            policy._obs_que.append(observation)
    return observation


def replanning():
    common = load_policies("hand_over", "joint", 100, 1, 2000, 8)
    policies = {k: load_policies("hand_over", "joint", 100, 1, 2000, k) for k in (1, 4)}
    policies[8] = common
    tapas = load_policies("hand_over", "tapas", 100, 1, 0, 0)
    env = environment("hand_over", "joint")
    try:
        observation = initialize(env, common)
        counter, original = simulation_counter(env)
        executed = 0
        with deadline(MAX_SECONDS), torch.inference_mode():
            while counter["steps"] < (5 if QUICK_TEST else 450):
                actions, _ = predict(common, observation, "joint")
                for index, action in enumerate(actions):
                    observation, _, _ = execute(env, action)
                    executed += 1
                    if index < len(actions) - 1:
                        common[0]._obs_que.append(observation)
                assert executed + 8 <= MAX_ACTIONS
            cached, _ = predict(common, observation, "joint")
            for k in (1, 4):
                policies[k][0].reset_episode(env)
                policies[k][0]._obs_que.extend(common[0]._obs_que)
            tapas[0].reset_episode(env)
            first_predictions = {}
            counted_step = env.task_env._pyrep.step
            def step(*args, **kwargs):
                value = counted_step(*args, **kwargs)
                if "tapas" not in first_predictions:
                    first_predictions["tapas"] = counter["steps"]
                    current = env.process_observation(env.task_env.get_observation())
                    predict(tapas, current, "tapas")
                    assert counter["steps"] == first_predictions["tapas"]
                return value
            env.task_env._pyrep.step = step
            force_step = counter["steps"]
            dt = env.task_env._pyrep.get_simulation_timestep()
            Shape("item0").add_force_and_torque(force=[0.05 / dt, 0, 0], torque=None)
            for index, action in enumerate(cached):
                observation, _, _ = execute(env, action)
                for k in (1, 4, 8):
                    if index + 1 == k:
                        first_predictions[k] = counter["steps"]
                        predict(policies[k], observation, "joint")
                        assert counter["steps"] == first_predictions[k]
                    elif index + 1 < k:
                        policies[k][0]._obs_que.append(observation)
            output = {"joint": [], "tapas": []}
            for key in (1, 4, 8, "tapas"):
                row = {
                    "episode": 0,
                    "force_applied_between_steps": [force_step, force_step + 1],
                    "first_prediction_step": first_predictions[key],
                    "delay_steps": first_predictions[key] - force_step,
                    "simulation_timestep_seconds": dt,
                }
                output["tapas" if key == "tapas" else "joint"].append(
                    row if key == "tapas" else {"k": key, **row})
        env.task_env._pyrep.step = original
    finally:
        env.close()
    return [save("replanning", "hand_over", model,
                 reference("replanning", "hand_over", model)["parameters"], rows)
            for model, rows in output.items()]

def plot_replanning(folder):
    import matplotlib.pyplot as plt
    from matplotlib.ticker import MaxNLocator
    labels, steps = [], []
    for model in ("joint", "tapas"):
        data = json.loads((folder / "replanning/hand_over" / f"{model}.json").read_text())
        for row in data["records"]:
            delay = row["first_prediction_step"] - row["force_applied_between_steps"][0]
            assert delay >= 0 and row.get("delay_steps", delay) == delay
            label = f"Joint K={row['k']}" if model == "joint" else "TAPAS"
            labels.append(f"{label}\n({data['parameters']['demos']} demos)")
            steps.append(delay)
    with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 15,
                         "axes.spines.top": False, "axes.spines.right": False,
                         "axes.edgecolor": "#777777", "axes.labelcolor": "#333333",
                         "xtick.color": "#333333", "ytick.color": "#333333"}):
        fig, ax = plt.subplots(figsize=(12, 6.4), dpi=220)
        bars = ax.bar(range(len(steps)), steps, color=["#2c94c8"] * (len(steps) - 1) + ["#bf841a"], width=0.54)
        ax.set_xticks(range(len(steps)), labels)
        ax.set_xlabel("Model", labelpad=13)
        ax.set_ylabel("Delay to next prediction [simulation steps]")
        ax.set_title("Handover", fontsize=18, pad=18)
        ax.set_ylim(0, max(max(steps), 1) * 1.25)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
        ax.tick_params(axis="x", pad=8)
        ax.grid(axis="y", color="#e7e7e7", linewidth=0.8)
        ax.set_axisbelow(True)
        ax.bar_label(bars, labels=[str(value) for value in steps], padding=5, fontsize=16)
        fig.subplots_adjust(left=0.135, right=0.95, bottom=0.18, top=0.86)
    return fig


### 10. Baseline

Test the models without offsets or impulses.

In [ ]:
run("baseline")


### 11. Constant action offset

Keep the same position offset for each arm during the whole episode.

In [ ]:
run("offset_constant")


### 12. Random action offset

Use a new random offset direction for each action.

In [ ]:
run("offset_random")


### 13. Shifted start positions

Move the objects to the edge of the demo start range or further out.

In [ ]:
run("position")


### 14. Impulse tests

Test how the models handle an impulse in Handover, Fridge and Oven.

In [ ]:
run("impulse")


### 15. Replanning test

Run the replanning test for Joint K=1, 4, 8 and TAPAS, then plot the delay in simulation steps.

In [ ]:
replanning()
replanning_figure = plot_replanning(OUTPUT)


### 16. Segmentation comparison

Run TAPAS with velocity-based segmentation. The gripper-based results come from the baseline.

In [ ]:
run("segmentation_comparison")


### 17. Demonstration signals

Save the gripper signals, positions and segment boundaries from the first Handover demo.

In [ ]:
def segmentation_signals():
    gripper = load_policies("hand_over", "tapas", 5, 1, 0, 0)[0].model
    velocity = load_policies("hand_over", "tapas_velocity", 5, 1, 0, 0)[0].model
    demos = gripper._demos
    env = environment("hand_over", "joint")
    try:
        env.reset()
        names = [obj.get_name() for obj, _ in env.task_env._task._initial_objs_in_scene]
        frame = demos.frame_names.index(f"obj{names.index('item0'):03d}")
    finally:
        env.close()
    parameters = {
        "demo": 0,
        "gripper_segments": [[int(s.start_idcs[0]), int(s.stop_idcs[0])] for s in gripper._demos_segments],
        "velocity_segments": [[int(s.start_idcs[0]), int(s.stop_idcs[0])] for s in velocity._demos_segments],
        "velocity_threshold_m_per_step": velocity.config.demos_segmentation.velocity_threshold,
    }
    records = []
    for step in range(demos.traj_lens[0]):
        records.append({
            "step": step,
            "left_gripper_state": int(demos.gripper_states_left[0][step].item()),
            "right_gripper_state": int(demos.gripper_states_right[0][step].item()),
            "left_ee_xyz_m": demos.left_ee_poses_raw[0][step, :3].tolist(),
            "right_ee_xyz_m": demos.right_ee_poses_raw[0][step, :3].tolist(),
            "cube_xyz_m": demos.frames2world_left[0][frame, step, :3, 3].tolist(),
        })
    return save("segmentation_signals", "hand_over", "demonstration", parameters, records)

segmentation_signals()


### 18. Time per action

Calculate time per action for the 100-demo Handover and Fridge tests. Show the average and standard deviation across seeds.

In [ ]:
import pandas as pd

timing_rows = []
for path in (OUTPUT / "baseline").glob("*/*.json"):
    data = json.loads(path.read_text())
    for record in data["records"]:
        timing_rows.append({"task": path.parent.name, "model": path.stem,
                            **data["parameters"], **record})
timing_data = pd.DataFrame(timing_rows).reindex(columns=[
    "task", "model", "demos", "epoch", "k", "seed", "episode", "status", "actions", "wall_seconds"])
timing_data = timing_data[
    timing_data["task"].isin(["hand_over", "put_bottle_in_fridge"]) &
    (timing_data["demos"] == 100) &
    timing_data["model"].isin(["joint", "tapas"]) &
    timing_data["epoch"].isin([0, 2000]) &
    (timing_data["actions"] > 0)
].copy()
timing_data["seconds_per_step"] = timing_data["wall_seconds"] / timing_data["actions"]
seed_times = timing_data.groupby(["model", "k", "seed", "task"])["seconds_per_step"].mean()
seed_times = seed_times.groupby(["model", "k", "seed"]).mean()
seed_times.groupby(["model", "k"]).agg(["mean", "std", "count"])
